## Part 3: SQL Analysis

This section performs SQL-based analysis on the cleaned datasets using joins, aggregations, window functions, CTEs, and subqueries to generate business insights such as revenue trends, customer segmentation, product performance, and cohort analysis.

### Step 1: Load Cleaned CSV Files into Spark DataFrames

In [0]:
# ==========================================
# Load Cleaned CSV Files
# ==========================================

orders_sql = spark.read.csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/orders_clean.csv",
    header=True,
    inferSchema=True
)

customers_sql = spark.read.csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/customers_clean.csv",
    header=True,
    inferSchema=True
)

products_sql = spark.read.csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/products_clean.csv",
    header=True,
    inferSchema=True
)

order_items_sql = spark.read.csv(
    "/Volumes/workspace/default/ecommerce_analytics_system/order_items_clean.csv",
    header=True,
    inferSchema=True
)

print("All cleaned datasets loaded successfully.")

All cleaned datasets loaded successfully.


### Step 2: Create Temporary SQL Views

In [0]:
# ==========================================
# Create Temporary SQL Views
# ==========================================

orders_sql.createOrReplaceTempView("orders")

customers_sql.createOrReplaceTempView("customers")

products_sql.createOrReplaceTempView("products")

order_items_sql.createOrReplaceTempView("order_items")

print("Temporary Views Created Successfully")

Temporary Views Created Successfully


### SQL Query 1 - Total Revenue per Category

In [0]:
%sql
SELECT
    p.category,

    ROUND(
        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100)
        ),
        2
    ) AS total_revenue

FROM order_items oi

JOIN products p
ON oi.product_id = p.product_id

GROUP BY p.category

ORDER BY total_revenue DESC;

category,total_revenue
Home,2.799080607E7
Books,2.636556263E7
Electronics,2.43023317E7
Clothing,2.261795009E7


### SQL Query 2 - Top 10 Customers by Total Order Value

In [0]:
%sql

SELECT

    c.customer_id,
    c.customer_name,

    ROUND(

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100)
        ),
        2
    ) AS total_order_value

FROM customers c

JOIN orders o
ON c.customer_id = o.customer_id

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY
    c.customer_id,
    c.customer_name

ORDER BY total_order_value DESC

LIMIT 10;

customer_id,customer_name,total_order_value
CUST0364,Xavier Shetty,1259752.49
CUST0344,Luke Biswas,1109277.37
CUST0336,Warda Loyal,1099211.78
CUST0170,Gaurika Bhatti,1002343.82
CUST0331,Krishna Sabharwal,986402.61
CUST0236,Edhitha Srinivasan,917623.96
CUST0397,Odika Tailor,910896.68
CUST0053,Rehaan Salvi,901392.76
CUST0495,Ekani Varkey,830629.5
CUST0413,Anya Venkatesh,804158.38


### SQL Query 3 - Month-wise Order Count for Last 12 Months

In [0]:
%sql

SELECT

DATE_FORMAT(order_date,'yyyy-MM') AS order_month,

COUNT(order_id) AS total_orders

FROM orders

WHERE order_date >= add_months(current_date(),-12)

GROUP BY DATE_FORMAT(order_date,'yyyy-MM')

ORDER BY order_month;

order_month,total_orders
2025-07,17
2025-08,32
2025-09,23
2025-10,37
2025-11,27
2025-12,27
2026-01,27
2026-02,27
2026-03,25
2026-04,31


### SQL Query 4 - Find Customers Who Placed Orders but Never Had Any Item Delivered

In [0]:
%sql

SELECT DISTINCT

c.customer_id,
c.customer_name

FROM customers c

JOIN orders o
ON c.customer_id = o.customer_id

WHERE c.customer_id NOT IN (

    SELECT customer_id

    FROM orders

    WHERE status = 'DELIVERED'

)

ORDER BY c.customer_id
LIMIT 10;

customer_id,customer_name
CUST0004,Pahal Oak
CUST0006,Farhan Memon
CUST0007,Fariq Kaul
CUST0009,Nihal Shere
CUST0010,Daksh Karnik
CUST0013,Imaran Dua
CUST0015,Radhika Dugar
CUST0018,Wriddhish Bhardwaj
CUST0020,Siddharth Zacharia
CUST0021,Zansi Seth


### SQL Query 5 - Products Ordered but Returned More Than Purchased

In [0]:
%sql

SELECT

    p.product_id,
    p.product_name,

    SUM(
        CASE
            WHEN oi.quantity > 0 THEN 1
            ELSE 0
        END
    ) AS purchases,

    SUM(
        CASE
            WHEN oi.quantity < 0 THEN 1
            ELSE 0
        END
    ) AS returns

FROM products p

JOIN order_items oi
ON p.product_id = oi.product_id

GROUP BY
    p.product_id,
    p.product_name

HAVING returns > purchases

ORDER BY returns DESC


product_id,product_name,purchases,returns
PROD0070,Velit Biography,1,2
PROD0322,Consectetur Education,0,1


### SQL Query 6 - Return Rate Per Category

In [0]:
%sql

SELECT

    p.category,

    COUNT(
        CASE
            WHEN oi.quantity < 0 THEN 1
        END
    ) AS returned_items,

    COUNT(*) AS total_items,

    ROUND(
        COUNT(
            CASE
                WHEN oi.quantity < 0 THEN 1
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS return_rate

FROM products p

JOIN order_items oi
ON p.product_id = oi.product_id

GROUP BY p.category

ORDER BY return_rate DESC;

category,returned_items,total_items,return_rate
Books,23,649,3.54
Clothing,17,557,3.05
Electronics,18,603,2.99
Home,17,691,2.46


### SQL Query 7 - Running Total of Revenue per Region

In [0]:
%sql

WITH daily_revenue AS (

    SELECT
        o.region_code,
        DATE(o.order_date) AS order_date,

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS daily_revenue

    FROM orders o

    JOIN order_items oi
    ON o.order_id = oi.order_id

    GROUP BY
        o.region_code,
        DATE(o.order_date)
)

SELECT
    region_code,
    order_date,

    ROUND(daily_revenue, 2) AS daily_revenue,

    ROUND(
        SUM(daily_revenue) OVER (
            PARTITION BY region_code
            ORDER BY order_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS running_total

FROM daily_revenue

ORDER BY
    region_code,
    order_date

LIMIT 10;

region_code,order_date,daily_revenue,running_total
EAST,2024-07-13,271879.74,271879.74
EAST,2024-07-18,54462.36,326342.10
EAST,2024-07-21,464846.56,791188.66
EAST,2024-07-29,4657.30,795845.96
EAST,2024-07-30,77982.00,873827.96
EAST,2024-08-02,162508.68,1036336.64
EAST,2024-08-12,285982.51,1322319.15
EAST,2024-08-22,131132.92,1453452.07
EAST,2024-08-31,165066.63,1618518.70
EAST,2024-09-02,45983.12,1664501.82


### SQL Query 8 - Rank Products by Revenue using DENSE_RANK

In [0]:
%sql

WITH product_revenue AS (

    SELECT

        p.category,
        p.product_name,

        ROUND(
            SUM(
                oi.quantity *
                oi.unit_price *
                (1 - oi.discount_percent / 100.0)
            ),
            2
        ) AS total_revenue

    FROM products p

    JOIN order_items oi
    ON p.product_id = oi.product_id

    GROUP BY
        p.category,
        p.product_name
)

SELECT

    category,
    product_name,
    total_revenue,

    DENSE_RANK() OVER (

        PARTITION BY category

        ORDER BY total_revenue DESC

    ) AS rank_in_category

FROM product_revenue

ORDER BY
    category,
    rank_in_category

LIMIT 10;

category,product_name,total_revenue,rank_in_category
Books,Dolorem Comics,834287.65,1
Books,Perspiciatis Comics,682902.51,2
Books,Placeat Novel,660380.58,3
Books,Id Novel,639150.96,4
Books,Quis Comics,588224.04,5
Books,Magni Comics,586949.33,6
Books,Inventore Comics,532373.71,7
Books,Necessitatibus Comics,524812.38,8
Books,Voluptatibus Comics,521791.42,9
Books,Earum Novel,476980.87,10


### SQL Query 9 - Days Between Consecutive Orders (LAG)

In [0]:
%sql

WITH order_gaps AS (

    SELECT
        customer_id,
        order_date,

        LAG(order_date) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS previous_order_date

    FROM orders
),

gap_data AS (

    SELECT
        customer_id,
        order_date,
        previous_order_date,

        DATEDIFF(
            order_date,
            previous_order_date
        ) AS days_gap

    FROM order_gaps
),

average_gap AS (

    SELECT
        customer_id,
        AVG(days_gap) AS avg_days_gap

    FROM gap_data

    WHERE days_gap IS NOT NULL

    GROUP BY customer_id
)

SELECT
    g.customer_id,
    g.order_date,
    g.previous_order_date,
    g.days_gap,

    ROUND(a.avg_days_gap, 2) AS average_gap,

    CASE
        WHEN a.avg_days_gap > 30 THEN 'At Risk'
        ELSE 'Active'
    END AS customer_status

FROM gap_data g

LEFT JOIN average_gap a
ON g.customer_id = a.customer_id

WHERE g.previous_order_date IS NOT NULL

ORDER BY
    g.customer_id,
    g.order_date

LIMIT 10;

customer_id,order_date,previous_order_date,days_gap,average_gap,customer_status
CUST0001,2025-11-29T05:52:56.400Z,2025-01-27T19:26:59.349Z,306,265.0,At Risk
CUST0001,2026-07-11T19:25:37.016Z,2025-11-29T05:52:56.400Z,224,265.0,At Risk
CUST0002,2025-07-14T09:19:58.155Z,2025-06-24T23:49:41.676Z,20,20.0,Active
CUST0005,2026-03-17T08:06:07.359Z,2025-06-25T22:04:16.615Z,265,265.0,At Risk
CUST0006,2026-06-10T18:01:33.621Z,2025-02-20T10:55:37.132Z,475,475.0,At Risk
CUST0007,2024-08-09T09:11:07.918Z,2024-07-18T02:09:11.094Z,22,350.0,At Risk
CUST0007,2026-06-18T02:44:59.237Z,2024-08-09T09:11:07.918Z,678,350.0,At Risk
CUST0011,2026-02-16T19:24:03.763Z,2024-12-04T23:32:17.371Z,439,439.0,At Risk
CUST0013,2026-06-25T00:31:09.468Z,2024-09-23T13:48:55.078Z,640,640.0,At Risk
CUST0015,2024-12-10T20:45:16.421Z,2024-12-07T10:21:48.118Z,3,3.0,Active


### SQL Query 10 - Monthly Revenue Category using Multiple CTEs

In [0]:
%sql

WITH monthly_customer_revenue AS (

    SELECT
        DATE_FORMAT(o.order_date, 'yyyy-MM') AS order_month,
        o.customer_id,

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS monthly_revenue

    FROM orders o

    JOIN order_items oi
    ON o.order_id = oi.order_id

    WHERE o.customer_id <> 'UNKNOWN'

    GROUP BY
        DATE_FORMAT(o.order_date, 'yyyy-MM'),
        o.customer_id
),

customer_segments AS (

    SELECT
        order_month,
        customer_id,
        monthly_revenue,

        CASE
            WHEN monthly_revenue > 10000 THEN 'High'
            WHEN monthly_revenue >= 5000 THEN 'Medium'
            ELSE 'Low'
        END AS customer_segment

    FROM monthly_customer_revenue
)

SELECT
    order_month,
    customer_segment,
    COUNT(customer_id) AS customer_count

FROM customer_segments

GROUP BY
    order_month,
    customer_segment

ORDER BY
    order_month,
    CASE
        WHEN customer_segment = 'High' THEN 1
        WHEN customer_segment = 'Medium' THEN 2
        ELSE 3
    END
    
    LIMIT 10;

order_month,customer_segment,customer_count
2024-01,High,2
2024-05,High,1
2024-07,High,16
2024-07,Low,2
2024-08,High,22
2024-09,High,26
2024-10,High,21
2024-10,Low,2
2024-11,High,31
2024-12,High,22


### SQL Query 11 - Customer Segmentation using NTILE

In [0]:
%sql

WITH customer_lifetime_value AS (

    SELECT
        o.customer_id,
        ROUND(
            SUM(
                oi.quantity *
                oi.unit_price *
                (1 - oi.discount_percent / 100.0)
            ),
            2
        ) AS total_value

    FROM orders o

    JOIN order_items oi
    ON o.order_id = oi.order_id

    WHERE o.customer_id <> 'UNKNOWN'

    GROUP BY o.customer_id
),

customer_quartiles AS (

    SELECT
        customer_id,
        total_value,
        NTILE(4) OVER (
            ORDER BY total_value DESC

        ) AS quartile

    FROM customer_lifetime_value
)

SELECT
    customer_id,
    total_value,
    quartile,
    CASE

        WHEN quartile = 1 THEN 'Platinum'
        WHEN quartile = 2 THEN 'Gold'
        WHEN quartile = 3 THEN 'Silver'
        ELSE 'Bronze'

    END AS quartile_label

FROM customer_quartiles

ORDER BY total_value DESC

LIMIT 10;

customer_id,total_value,quartile,quartile_label
CUST0364,1259752.49,1,Platinum
CUST0344,1109277.37,1,Platinum
CUST0336,1099211.78,1,Platinum
CUST0170,1002343.82,1,Platinum
CUST0331,986402.61,1,Platinum
CUST0236,917623.96,1,Platinum
CUST0397,910896.68,1,Platinum
CUST0053,901392.76,1,Platinum
CUST0495,830629.50,1,Platinum
CUST0413,804158.38,1,Platinum


### SQL Query 12 - Year-over-Year Revenue Comparison

In [0]:
%sql

WITH monthly_revenue AS (

    SELECT
        YEAR(o.order_date) AS year,
        MONTH(o.order_date) AS month,

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS revenue

    FROM orders o

    JOIN order_items oi
    ON o.order_id = oi.order_id

    GROUP BY
        YEAR(o.order_date),
        MONTH(o.order_date)
),

year_comparison AS (

    SELECT
        current_year.year,
        current_year.month,
        current_year.revenue,

        previous_year.revenue AS prev_year_revenue

    FROM monthly_revenue current_year

    LEFT JOIN monthly_revenue previous_year
    ON current_year.month = previous_year.month
    AND current_year.year = previous_year.year + 1
)

SELECT
    year,
    month,

    ROUND(revenue, 2) AS revenue,

    ROUND(prev_year_revenue, 2) AS prev_year_revenue,

    CASE
        WHEN prev_year_revenue IS NULL
             OR prev_year_revenue = 0
        THEN NULL

        ELSE ROUND(
            (
                (revenue - prev_year_revenue)
                / prev_year_revenue
            ) * 100,
            2
        )
    END AS yoy_growth_percent

FROM year_comparison

ORDER BY
    year,
    month
    
    LIMIT 10;

year,month,revenue,prev_year_revenue,yoy_growth_percent
2024,1,167284.59,null,null
2024,5,157435.33,null,null
2024,7,2772915.32,null,null
2024,8,3717162.96,null,null
2024,9,3764555.27,null,null
2024,10,3794838.89,null,null
2024,11,5852002.98,null,null
2024,12,4322884.02,null,null
2025,1,4437598.79,167284.59,2552.72
2025,2,3495897.61,null,null


### SQL Query 13 - First Purchased Category vs Latest Purchased Category

In [0]:
%sql

WITH customer_categories AS (

    SELECT
        o.customer_id,
        p.category,
        o.order_date,
        o.order_id,

        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date, o.order_id
        ) AS first_order,

        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date DESC, o.order_id DESC
        ) AS last_order

    FROM orders o

    JOIN order_items oi
        ON o.order_id = oi.order_id

    JOIN products p
        ON oi.product_id = p.product_id

    WHERE o.customer_id <> 'UNKNOWN'
)

SELECT
    f.customer_id,
    f.category AS first_category,
    l.category AS last_category,

    CASE
        WHEN f.category = l.category THEN 'No'
        ELSE 'Yes'
    END AS category_shift

FROM customer_categories f

JOIN customer_categories l
    ON f.customer_id = l.customer_id

WHERE
    f.first_order = 1
    AND l.last_order = 1

ORDER BY f.customer_id

LIMIT 10;

customer_id,first_category,last_category,category_shift
CUST0001,Home,Clothing,Yes
CUST0002,Home,Clothing,Yes
CUST0004,Electronics,Electronics,No
CUST0005,Clothing,Books,Yes
CUST0006,Clothing,Home,Yes
CUST0007,Electronics,Books,Yes
CUST0008,Books,Books,No
CUST0009,Books,Books,No
CUST0010,Books,Books,No
CUST0011,Books,Books,No


### SQL Query 14 - Cumulative Revenue Distribution

In [0]:
%sql

WITH customer_revenue AS (

    SELECT
        o.customer_id,

        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ) AS revenue

    FROM orders o

    JOIN order_items oi
    ON o.order_id = oi.order_id

    WHERE o.customer_id <> 'UNKNOWN'

    GROUP BY o.customer_id
)

SELECT
    customer_id,

    ROUND(revenue, 2) AS revenue,

    ROUND(
        SUM(revenue) OVER (
            ORDER BY revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ),
        2
    ) AS cumulative_revenue,

    ROUND(
        SUM(revenue) OVER (
            ORDER BY revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        )
        /
        SUM(revenue) OVER ()
        * 100,
        2
    ) AS cumulative_percent

FROM customer_revenue

ORDER BY revenue DESC

LIMIT 10;

customer_id,revenue,cumulative_revenue,cumulative_percent
CUST0364,1259752.49,1259752.49,1.30
CUST0344,1109277.37,2369029.86,2.45
CUST0336,1099211.78,3468241.64,3.59
CUST0170,1002343.82,4470585.46,4.62
CUST0331,986402.61,5456988.07,5.64
CUST0236,917623.96,6374612.03,6.59
CUST0397,910896.68,7285508.71,7.54
CUST0053,901392.76,8186901.47,8.47
CUST0495,830629.50,9017530.97,9.33
CUST0413,804158.38,9821689.35,10.16


### SQL Query 15 - Customer Cohort Analysis

In [0]:
%sql

WITH customer_cohort AS (

    SELECT
        customer_id,
        DATE_TRUNC(
            'MONTH',
            TO_DATE(registration_date)
        ) AS cohort_month

    FROM customers
),

customer_activity AS (

    SELECT DISTINCT
        cc.customer_id,
        cc.cohort_month,

        CAST(
            MONTHS_BETWEEN(
                DATE_TRUNC('MONTH', o.order_date),
                cc.cohort_month
            ) AS INT
        ) AS month_number

    FROM customer_cohort cc

    LEFT JOIN orders o
        ON cc.customer_id = o.customer_id
),

cohort_summary AS (

    SELECT
        cohort_month,

        COUNT(DISTINCT customer_id) AS total_customers,

        COUNT(DISTINCT CASE
            WHEN month_number = 0 THEN customer_id
        END) AS month_0,

        COUNT(DISTINCT CASE
            WHEN month_number = 1 THEN customer_id
        END) AS month_1,

        COUNT(DISTINCT CASE
            WHEN month_number = 2 THEN customer_id
        END) AS month_2,

        COUNT(DISTINCT CASE
            WHEN month_number = 3 THEN customer_id
        END) AS month_3

    FROM customer_activity

    GROUP BY cohort_month
)

SELECT
    DATE_FORMAT(cohort_month, 'yyyy-MM') AS cohort_month,

    total_customers,
    month_0,
    month_1,
    month_2,
    month_3,

    ROUND(
        month_0 * 100.0 / total_customers,
        2
    ) AS month_0_retention_rate,

    ROUND(
        month_1 * 100.0 / total_customers,
        2
    ) AS month_1_retention_rate,

    ROUND(
        month_2 * 100.0 / total_customers,
        2
    ) AS month_2_retention_rate,

    ROUND(
        month_3 * 100.0 / total_customers,
        2
    ) AS month_3_retention_rate

FROM cohort_summary

WHERE month_0 > 0
   OR month_1 > 0
   OR month_2 > 0
   OR month_3 > 0

ORDER BY cohort_month

LIMIT 10;

cohort_month,total_customers,month_0,month_1,month_2,month_3,month_0_retention_rate,month_1_retention_rate,month_2_retention_rate,month_3_retention_rate
2024-06,23,0,3,1,0,0.00,13.04,4.35,0.00
2024-08,14,0,1,0,2,0.00,7.14,0.00,14.29
2024-09,15,1,0,0,1,6.67,0.00,0.00,6.67
2024-10,9,1,0,0,2,11.11,0.00,0.00,22.22
2024-11,21,1,2,1,1,4.76,9.52,4.76,4.76
2024-12,15,1,3,2,0,6.67,20.00,13.33,0.00
2025-01,10,0,1,1,0,0.00,10.00,10.00,0.00
2025-02,14,4,0,0,0,28.57,0.00,0.00,0.00
2025-03,11,2,0,0,0,18.18,0.00,0.00,0.00
2025-05,8,0,1,0,0,0.00,12.50,0.00,0.00


### SQL Query 16 - Products Frequently Bought Together

In [0]:
%sql

SELECT

    a.product_id AS product_a,

    b.product_id AS product_b,

    COUNT(*) AS times_bought_together

FROM order_items a

JOIN order_items b

ON a.order_id = b.order_id

AND a.product_id < b.product_id

GROUP BY

    a.product_id,

    b.product_id

ORDER BY

    times_bought_together DESC

LIMIT 10;

product_a,product_b,times_bought_together
PROD0073,PROD0484,3
PROD0321,PROD0498,2
PROD0085,PROD0349,2
PROD0063,PROD0124,2
PROD0142,PROD0349,2
PROD0050,PROD0164,2
PROD0434,PROD0485,2
PROD0076,PROD0270,2
PROD0018,PROD0149,2
PROD0083,PROD0149,2
